In [ ]:
# from doggopyr.tools.helper_functions import Module as hf


## 🛠️ The Senior Data Engineer Assessment: Marketing Attribution Pipeline

### **Scenario**

<p style='color:lightgreen'>A major client needs a robust data platform to analyze digital marketing campaign performance. The core task is to process
<span style='font-weight:bold; color:yellow'>massive clickstream data</span>
(web logs) and <span style='font-weight:bold; color:yellow'>impression data</span>
(ad views) and sales data (ERP/CRM) to create a daily <span style='font-weight:bold; color:yellow'>Marketing Attribution Model</span>
showing which campaigns led to customer purchases.

The expected volume is: <span style='font-weight:bold; color:yellow'>100 Billion</span>
raw events per day, needing to be processed and loaded into a final data warehouse for reporting.
</p>
-----

## 📝 <span style='font-weight:bold; color:pink'>Part 1: System Design & Architecture (50% Weight)</span>

Design the end-to-end architecture to meet the following requirements:

1.  <span style='font-weight:bold; color:yellow'>Ingestion:</span> Must handle a mix of high-volume, continuous streams (click data) and daily batched loads (purchase data).


2.  <span style='font-weight:bold; color:yellow'>Processing:</span> Must use **AWS data integration and processing technologies (Glue, Athena, Redshift)** and **Spark**. The pipeline must ensure data is **de-duplicated** and joined before loading.


3.  <span style='font-weight:bold; color:yellow'>Storage:</span> Implement a **Lakehouse architecture** using **Amazon S3** as the foundational Data Lake storage.


4.  <span style='font-weight:bold; color:yellow'>Consumption:</span> The final, aggregated attribution table must reside in **Amazon Redshift** for fast analytical querying.


### <span style='font-weight:bold; color:pink'>Task 1.1: Component Selection and Flow</span>



---------------------------------- OVERVIEW ------------------------------------
<body>
<div style='color:white;background-color:navyblue'>
The source of the streaming data is a data broker tool like apache kafka or rabbit mq.
If the source is apache kafka we would preferably use AWS Kinesis Firehose since this service 
<br>
is highly optimised and does the loading of the stream to the bronze layer end to end.
<br>
If the stream broker uses a different protocol (like rabbit mq does) we would
use a connector from the Kafka Connect framework. This framework has ready made connectors
that convert stream formats into a Kafka stream format.
<br>
<b>This is known as a Kafka bridge connector.</b>
<br>If the clicks come from a standard database or a message queue we would turn this data into an incremental stream by using a CDC connector
<br>that opens a websocket that pulls any new insertion or change to a record in the target database. This
turns all external services into data stream sources.

Glue is Amazon's ETL tool that is built on top of Apache Spark. It creates the environment to operate pyspark scripts as jobs.

Our S3 bucket would be divided into 3 layers: raw (bronze), staging (silver)
and processed (gold).

The ingestion process would include a connector to the data sources
(marketing api, sales db, click stream broker).
The connectors would be configured to load the data in it's raw format
(json/yaml/csv/xml etc) into the bronze layer of the the bucket and
put a timestamp in the file name for easier pull.
=> That way we would have a persistent storage for historical data.

<span style='font-weight:bold; color:cyan'>Bronze (raw) layer</span>,
we would create three separate pipelines:
<li>One for daily marketing batches (Glue - API connnector / custom connector)
<li> One for clicks stream data (AWS Kinesis Firehose / Kafka connect).
<li> And the last for sales data (Glue - JDBC connector).

<br>
<li>For sales data:
<li>If the sales data is stored in one of our databases we would use a simple jdbc
connection url.
<li>Then we would use a glue connector to pull the data into an S3 folder
<pre>
  (folders in S3 are basically a namespace, all files within a folder have a common segment in their paths)
</pre>
<li>If the sales data is not available usign a JDBC connection,
<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;we would create a custom connector in python and import it by configuring
the glue job. The connector would send a request to whichever api is avalilable
<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;and would pull data for the current date and if needed also for a while back.
<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;For instance 30 days back.
The files we pull with the connector would be in their rawest form, json, csv,
xml, yaml etc.
</li>
<br>
<li>For click streams:
<list>
<li>
We would use a streaming connector to pull data from the streaming service (data broker).
Preferably AWS Kinesis Firehose (if available and the streaming service is Kafka).
<br>
<li>Else, we would try to find a connector in the Kafka Connect framework that would convert
the stream format to a format supported by kafka, use it as a bridge (adapter)
<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;and connect its output to Firehose.
<br>
<li>If Kinesis Firehose is not available we would use a Glue pyspark custom stream connector
though this is more costly and less effective.
</list>
<br>
<br>
<br>
<li>For marketing data (impressions):
<list>
<li>
We would use a Glue connector to the ads api.
</li>
<li>
If one does not exist, we would create a pyspark job that connects to the ads 
api, queries the api, pulls the data into a pyspark dataframe and writes 
the data as a csv/json to the bronze layer in our S3 bucket in a folder
corresponding to this specific marketing api (Google Ads for instance).
</li>
</list>

<span style='font-weight:bold; color:cyan'>Silver (staging) layer:</span>
<br>
<br>
<b>NOTE:</b> We would configure our Glue component to sink the raw data in separate folder for 
each source (clicks, sales, impressions) and create separate files for each date.
<br>
For the stream data we would create a subfolder for each date and partition
the files for different hours to make the pulling of data easier.
<br>
<br>


<list> Daily marketing batches (ads impressions):
<li>Assuming we want to track the behaviour of customers by their user id,
we would use a window function to deduplicate records.
<li>We would group the data by user ids (email or any other identifier).
<li>We make sure to normalise the user ids and timestamps so they would be of the same form across the silver schema.
<li>Then we would order by insertion timestamp of records within that group in a descending order and assign a row
number within the group.
<li>Then drop all records except the first row = latest.
<li>Finally we would sink the data (write) to an S3 bucket dedicated for the staging layer using pyspark's built in
write method.
</list>

<list> Daily sales data:
<li>Similarily to the marketing data, we would group the data by user ids (email or any other identifier).
<li>We make sure to normalise the user ids and timestamps so they would be of the same form across the silver schema.
<li>Then we would use a window function to deduplicate records (we allow
duplicates in the bronze layer) for the same user/buyer and the same sales record id.
<li>Finally we would sink the data (write) to an S3 bucket dedicated for the
staging layer.
</list>

<br>
<list> Clicks stream:
<li>Now that we have dumps of click logs, we can deduplicate the log lines that have the exact same timestamp per
<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;user per event type (using a window function with a deduplication key (user_id, timestamp, event_type)).
<li>We make sure to normalise the user ids and timestamps so they would be of the same form across the silver schema.
<li> Then, we would group the data by user id and order by timestamp.
</list>

<br>
<br>
<b>NOTE:</b>
<br>
<b>Bronze layer</b>: ingestion,
<br>
<b>Silver layer</b>: clean and deduplicated data but separate for each source = SSOT (single source of truth) for each source
<br>
<b>Gold layer</b>: After ATTRIBUTE JOIN - records from different sources are joined together by a common attribute (user id in this case).
<br>
<br>
<span style='font-weight:bold; color:cyan'>For the final layer:</span>
<br>

<list>
<li> A pyspark job in Glue that loads the marketing, sales and clicks data 30 days back.
<li> The data would be loaded into three separate pyspark dfs.
<li> We order the clicks dataframe by timestamp (ascending order).
<li> We join the three dfs using the user id and ordering by user id and timestamp (asc).
<li> We should now have the full story of every user in a chronological order.
</list>

<span style='font-weight:bold; color:cyan'>Loading the aggregated data to Redshift:</span>
<li> A pyspark job that pulls the curated data from the S3 gold layer, creates 
a target table in Redshift if it doesn't exist, adds a serial id and a modifiedat
<br>columns + trigger to update the modifiedat column (if one or more of these columns does not already exist).
<br>Then the job inserts the data to the Redshift table.
<br>
<b>NOTE:</b>
Since Spark is built on RDDs we don't need to manually lazy load the data and perform bulk inserts.
Spark would insert the data to redshipt partition by partition and manage the load.
<br>
</div>
</body>
---------------------------------- OVERVIEW ------------------------------------



---
1.  **Draw/Describe the architecture diagram.** Use specific AWS services mentioned (S3, Glue, Athena, Redshift).
2.  **Describe the role of each service in the pipeline:**
      * **Click Stream:** How is it ingested and staged in S3?
      * **Data Processing:** How is **AWS Glue (Spark)** used to read data from the S3 Lake, perform the attribution logic (join clicks/impressions to purchases), and write the result back?
      * **Data Loading:** How is the final structured data efficiently loaded from S3 into **Redshift**? (Hint: Consider COPY command or Glue's optimized Redshift connector).



---
### Task 1.2: Lakehouse Design and Data Governance

1.  **S3 Layering:** Define the three layers you would use in your S3 data lake (e.g., Raw, Staging/Silver, Curated/Gold). Justify the purpose of each.
2.  **Partitioning Strategy:** For the clickstream data (100 Billion events/day), recommend the optimal **partitioning strategy** (e.g., by date, time, or client ID). Explain *how* this strategy enhances performance in both **Spark (Glue)** processing and **Athena** querying.

-----



## 💻 Part 2: Spark/Python Optimization & Data Structures (30% Weight)

This task focuses on performance and your understanding of Spark's inner workings.

### Task 2.1: Spark Optimization and Joins

You have two Spark DataFrames:

  * `df_clicks`: 100 Billion records.
  * `df_campaigns`: **10,000 records** (metadata about the campaigns).

You need to enrich `df_clicks` by joining it to `df_campaigns` on the `campaign_id` key.

1.  **Identify the Optimal Join Strategy:** Given the huge size difference,
what is the most efficient type of Spark join to perform this enrichment?
2.  **Explain the Mechanism:** Describe the underlying concept/mechanism
(involving a specific **data structure**) that makes this join strategy faster
than a standard Sort-Merge Join. Explain why the average lookup time is near
**$O(1)$** for the larger DataFrame.
      * *(Hint: The answer should mention Hash Tables and memory distribution.)*



### Task 2.2: Python Generator vs. List (Conceptual)

Explain how a **Python Generator** (similar to a Linked List concept) is more
memory efficient than a standard Python **List** when dealing with reading a
massive, multi-gigabyte source file before loading it into Spark.

-----



## 📊 Part 3: SQL & Data Modeling (20% Weight)

The final requirement is a denormalized table in **Redshift** optimized for reporting.

### Task 3.1: Redshift Distribution and Sort Keys

You are defining the final, aggregated attribution table in Redshift: `ATTRIBUTION_SUMMARY`.

| Column Name | Data Type | Purpose |
| :--- | :--- | :--- |
| `reporting_date` | DATE | Used by all reports for time-series filtering. |
| `campaign_id` | INT | Used frequently for joining to Campaign Dimension Tables. |
| `user_id` | BIGINT | Used for detailed user analysis. |
| `conversion_count` | INT | The metric for reporting. |

1.  **Distribution Key (DISTKEY):** Which column would you choose as the `
DISTKEY`? Justify your choice based on how Redshift stores and processes data (i.e., minimizing data movement during joins).
2.  **Sort Key (SORTKEY):** Which column(s) would you choose as the `SORTKEY`?
Justify why this helps reporting queries run faster.



### Task 3.2: De-Duplication and Idempotency (SQL)

The raw clickstream data contains duplicates. Write a **SQL query** (or a
common table expression/CTE) that selects the **most recent unique record** for each `(user_id, click_timestamp)` combination, based on a single `event_id` column.

  * *(Hint: Use a Window Function like `ROW_NUMBER()` or `RANK()`).*

<!-- end list -->

```sql
-- Write your SQL solution here to find the latest unique click record.
-- Assume the table is named RAW_CLICKS.
```
